# LINMA1702 — Optimisation barrage hydroélectrique
## Partie 1

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cvxpy as cp
from datetime import datetime, timedelta

---
## Utilities fonctions


In [2]:
DEFAULT_SOLVER = cp.HIGHS
DELTA_T = 1.0

def parse_scenario(filepath: str):
    """Parse a scenario .txt file into a parameter dictionary."""
    with open(filepath, "r") as fh:
        content = fh.read()

    tokens = content.split()
    params = {}
    i = 0
    current_key = None
    vector_buffer = []

    SCALAR_KEYS = {"N", "V0", "Vmin", "Vmax", "Tmax", "Dmax", "Mmax",
                   "ET", "ME", "TDmin", "VTmax", "VDmax"}
    VECTOR_KEYS = {"F", "P"}

    while i < len(tokens):
        token = tokens[i]

        if token in SCALAR_KEYS:
            # flush any pending vector
            if current_key in VECTOR_KEYS and vector_buffer:
                params[current_key] = np.array(vector_buffer, dtype=float)
                vector_buffer = []

            current_key = token
            params[current_key] = float(tokens[i + 1])
            i += 2

        elif token in VECTOR_KEYS:
            # flush any pending vector
            if current_key in VECTOR_KEYS and vector_buffer:
                params[current_key] = np.array(vector_buffer, dtype=float)
                vector_buffer = []

            current_key = token
            i += 1

        else:
            # accumulate numeric values for the current vector key
            if current_key in VECTOR_KEYS:
                vector_buffer.append(float(token))
            i += 1

    # flush last vector
    if current_key in VECTOR_KEYS and vector_buffer:
        params[current_key] = np.array(vector_buffer, dtype=float)

    params["N"] = int(params["N"])
    return params

def _build_variables(N: int):
    """Declare all CVXPY decision variables."""
    T = cp.Variable(N, name="T", nonneg=True)   # turbined flow   
    D = cp.Variable(N, name="D", nonneg=True)   # spillage flow   
    M = cp.Variable(N, name="M", nonneg=True)   # pumped flow     
    V = cp.Variable(N + 1, name="V", nonneg=True)  # reservoir vol
    return T, D, M, V

def _build_constraints(T, D, M, V, params: dict):
    """Assemble all linear constraints."""
    N     = params["N"]
    F     = params["F"]
    V0    = params["V0"]
    Vmin  = params["Vmin"]
    Vmax  = params["Vmax"]
    Tmax  = params["Tmax"]
    Dmax  = params["Dmax"]
    Mmax  = params["Mmax"]
    TDmin = params["TDmin"]
    VTmax = params["VTmax"]
    VDmax = params["VDmax"]
    dt    = DELTA_T

    c_V0    = V[0] == V0
    c_VN    = V[N] == V0
    c_Vmin  = V >= Vmin
    c_Vmax  = V <= Vmax
    c_Tmax  = T <= Tmax
    c_Dmax  = D <= Dmax
    c_Mmax  = M <= Mmax
    c_TDmin = T + D >= TDmin
    dynamics = [V[k+1] == V[k] + (F[k] + M[k] - T[k] - D[k]) * dt for k in range(N)]
    ramp_T   = [c for k in range(1, N) for c in [
                    T[k] - T[k-1] <=  VTmax * dt,
                    T[k-1] - T[k] <=  VTmax * dt]]
    ramp_D   = [c for k in range(1, N) for c in [
                    D[k] - D[k-1] <=  VDmax * dt,
                    D[k-1] - D[k] <=  VDmax * dt]]

    named = {
        "c_V0": c_V0, "c_VN": c_VN,
        "c_Vmin": c_Vmin, "c_Vmax": c_Vmax,
        "c_Tmax": c_Tmax, "c_Dmax": c_Dmax,
        "c_Mmax": c_Mmax, "c_TDmin": c_TDmin,
        "ramp_T": ramp_T, "ramp_D": ramp_D,
        "dynamics": dynamics,
    }
    all_constraints = ([c_V0, c_VN, c_Vmin, c_Vmax, c_Tmax, c_Dmax,
                        c_Mmax, c_TDmin] + dynamics + ramp_T + ramp_D)
    return all_constraints, named

def _build_objective(T, D, M, params: dict):
    """Revenue = sum_k P[k] * (ET * T[k] - ME * M[k]) * dt"""
    P  = params["P"]
    ET = params["ET"]
    ME = params["ME"]
    dt = DELTA_T

    revenue = cp.sum(cp.multiply(P, ET * T - ME * M)) * dt
    return revenue


---
## Extraction des données

In [3]:
params1 = parse_scenario("../data/BelgiumScenario1.txt")
params2 = parse_scenario("../data/BelgiumScenario2.txt")

N1 = params1["N"]
P1 = params1["P"]
F1 = params1["F"]

N2 = params2["N"]
P2 = params2["P"]
F2 = params2["F"]

# Build time axis starting 1 June
t0 = datetime(2024, 6, 1)
times1 = [t0 + timedelta(hours=k) for k in range(N1)]
times2 = [t0 + timedelta(hours=k) for k in range(N2)]


---
## Q1.4 - Fonction Hydro

In [4]:
def hydro(data):
    """
    Calcul de la solution optimale du problème

    Paramètres
    ----------
    data : string
        Chemin vers un fichier de scénario (par exemple "BelgiumScenario1.txt").

    Returns
    -------
    sol : dictionnaire 
    """
    params = parse_scenario(data)
    N = params["N"]

    T, D, M, V = _build_variables(N)
    constraints, reference = _build_constraints(T, D, M, V, params)
    objective   = _build_objective(T, D, M, params)

    problem = cp.Problem(cp.Maximize(objective), constraints)
    problem.solve(solver=DEFAULT_SOLVER, verbose=False)

    sol = {
        "V":      V.value,
        "T":      T.value,
        "D":      D.value,
        "M":      M.value,
        "valopt": problem.value,
        
        ##Complementaires pour analyse post-solution
        "problem": problem,
        "params":  params,
        "reference": reference,
    }
    return sol

---
## Q1.5 — Solution optimal et comparaison avec la stratégie de ref

In [5]:
sol = hydro("../data/BelgiumScenario1.txt")
print(f"Valeur optimale (avec pompage)  : {sol['valopt']:,.0f} euros")

Valeur optimale (avec pompage)  : 11,083,291 euros


In [6]:
def reference_strategy_revenue(params):
    """
    Revenue de la stratégie de référence sans pompage:
    T[k] = F[k] à cahque instant
    D[k] = 0, M[k] = 0.
    """
    P  = params["P"]
    F  = params["F"]
    ET = params["ET"]
    dt = DELTA_T
    return float(np.sum(P * ET * F) * dt)

In [7]:
def no_pump_revenue(sol):
    """Revenue de la solution optimale mais sans pompage."""
    params_no_pump = dict(sol["params"])
    params_no_pump["Mmax"] = 0.0

    N = params_no_pump["N"]
    T, D, M, V = _build_variables(N)
    constraints, _ = _build_constraints(T, D, M, V, params_no_pump)
    objective   = _build_objective(T, D, M, params_no_pump)

    prob = cp.Problem(cp.Maximize(objective), constraints)
    prob.solve(solver=DEFAULT_SOLVER, verbose=False)
    return prob.value

In [8]:
rev_ref     = reference_strategy_revenue(sol["params"])
rev_no_pump = no_pump_revenue(sol)
rev_opt     = sol["valopt"]

print(f"Stratégie de référence (T=F, M_k = 0)      : {rev_ref:,.0f} euros")
print(f"Optimisation sans pompage  (M_k = 0)       : {rev_no_pump:,.0f} euros")
print(f"Optimisation avec pompage                  : {rev_opt:,.0f} euros")
print()
print(f"Gain de l'optimisation                     : {rev_opt - rev_ref:,.0f} euros")
print(f"Gain spécifique du pompage                 : {rev_opt - rev_no_pump:,.0f} euros")

Stratégie de référence (T=F, M_k = 0)      : 8,119,554 euros
Optimisation sans pompage  (M_k = 0)       : 9,589,005 euros
Optimisation avec pompage                  : 11,083,291 euros

Gain de l'optimisation                     : 2,963,737 euros
Gain spécifique du pompage                 : 1,494,286 euros


---
## Q1.6 — Analyse de sensibilité (prix duaux)

In [9]:
constraint_reference = sol["reference"]

dual_Vmax  = np.sum(constraint_reference["c_Vmax"].dual_value)
dual_Tmax  = np.sum(constraint_reference["c_Tmax"].dual_value)
dual_Mmax  = np.sum(constraint_reference["c_Mmax"].dual_value)
dual_VTmax = np.sum([c.dual_value for c in constraint_reference["ramp_T"]])

print(f"Valeur optimale : {sol['valopt']:,.0f} euros\n")
print("Sensibilités (valeurs duales agrégées) :")
print(f"  dOpt/dVmax  = {dual_Vmax:.4f} euros/m3")
print(f"  dOpt/dTmax  = {dual_Tmax:.4f} euros/(m3/h)")
print(f"  dOpt/dMmax  = {dual_Mmax:.4f} euros/(m3/h)")
print(f"  dOpt/dVTmax = {dual_VTmax:.4f} euros*h/m3")

Valeur optimale : 11,083,291 euros

Sensibilités (valeurs duales agrégées) :
  dOpt/dVmax  = 0.5650 euros/m3
  dOpt/dTmax  = 3.0147 euros/(m3/h)
  dOpt/dMmax  = 0.9903 euros/(m3/h)
  dOpt/dVTmax = 0.6871 euros*h/m3


In [10]:
ET = sol["params"]["ET"]
dt = DELTA_T

dual_F = np.array([c.dual_value for c in constraint_reference["dynamics"]])
dual_P = ET * sol["T"] * dt

print("\nSensibilité à F[k] (prix de l'eau, euros/m3) :")
print(f"  min={dual_F.min():.4f}  mean={dual_F.mean():.4f}  max={dual_F.max():.4f}")
print("\nSensibilité à P[k] (euros par unité de euros/MWh) :")
print(f"  min={dual_P.min():.2f}  mean={dual_P.mean():.2f}  max={dual_P.max():.2f}")


Sensibilité à F[k] (prix de l'eau, euros/m3) :
  min=-0.0095  mean=0.0565  max=0.0848

Sensibilité à P[k] (euros par unité de euros/MWh) :
  min=0.00  mean=34.49  max=80.00


In [11]:
DELTA_VMAX = 500_000  # plutot grande perturbation

dual_Vmax = np.sum(sol["reference"]["c_Vmax"].dual_value)
linear_estimate = dual_Vmax * DELTA_VMAX

params_perturbed = dict(sol["params"])
params_perturbed["Vmax"] += DELTA_VMAX

# Re-solve via fonction auxiliaire
def resolve_with_params(params):
    N = params["N"]
    T, D, M, V = _build_variables(N)
    constraints, _ = _build_constraints(T, D, M, V, params)
    objective = _build_objective(T, D, M, params)
    prob = cp.Problem(cp.Maximize(objective), constraints)
    prob.solve(solver=DEFAULT_SOLVER, verbose=False)
    return prob.value

val_perturbed = resolve_with_params(params_perturbed)
real_gain = val_perturbed - sol["valopt"]

print(f"dVmax = {DELTA_VMAX:,} m3")
print(f"Estimation linéaire : {linear_estimate:,.0f} euros")
print(f"Gain réel (re-optimisation) : {real_gain:,.0f} euros")
print(f"Erreur relative : {abs(real_gain - linear_estimate)/abs(real_gain)*100:.1f} %")

dVmax = 500,000 m3
Estimation linéaire : 282,510 euros
Gain réel (re-optimisation) : 256,097 euros
Erreur relative : 10.3 %
